In [0]:
USE e_commerce.silver;

In [0]:
%run "../src/utils/merge"

In [0]:
CREATE TABLE IF NOT EXISTS silver.t_order_x_product
USING DELTA
LOCATION 'abfss://silver@saexternaldatabricks.dfs.core.windows.net/t_order_x_product'
AS
SELECT
  oi.order_id,
  oi.order_item_id,
  oi.status,
  oi.purchase_date,
  oi.approved_date,
  oi.carrier_date,
  oi.delivered_customer_date,
  oi.estimated_delivery_date,
  oi.customer_id,
  oi.seller_id,
  oi.shipping_limit_date,
  oi.price,
  oi.freight_value,
  p.product_id,
  p.category_name,
  p.name_lenght,
  p.description_lenght,
  p.photos_qty,
  p.weight_g,
  p.length_cm,
  p.height_cm,
  p.width_cm,
  current_timestamp() as ingestion_timestamp
FROM
  t_order_x_item oi
INNER JOIN
  silver.product p
ON
  oi.product_id = p.product_id
LIMIT 0;

In [0]:
OPTIMIZE silver.t_order_x_product
ZORDER BY (order_id, order_item_id);

In [0]:
%python
df_order_x_product_updates = spark.sql("""
    SELECT
      oi.order_id,
      oi.order_item_id,
      oi.status,
      oi.purchase_date,
      oi.approved_date,
      oi.carrier_date,
      oi.delivered_customer_date,
      oi.estimated_delivery_date,
      oi.customer_id,
      oi.seller_id,
      oi.shipping_limit_date,
      oi.price,
      oi.freight_value,
      p.product_id,
      p.category_name,
      p.name_lenght,
      p.description_lenght,
      p.photos_qty,
      p.weight_g,
      p.length_cm,
      p.height_cm,
      p.width_cm,
      current_timestamp() as ingestion_timestamp
FROM
  t_order_x_item oi
INNER JOIN
  silver.product p
ON
  oi.product_id = p.product_id
""")

In [0]:
%python
try:
    merge(
        df_source = df_order_x_product_updates,
        target_table = "e_commerce.silver.t_order_x_product",
        merge_key = ["order_id","order_item_id"],
        transforms = {

        },
        exclude_update_cols = []
    )
except Exception as e:
    print(e)

In [0]:
SELECT COUNT(1) FROM silver.t_order_x_product;